In [0]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

spark = SparkSession.builder.getOrCreate()

# ---------------------------------------------------------
# 1. LOAD PARQUET FROM DBFS
# ---------------------------------------------------------
# Upload your file to /FileStore/ first via Data > Add Data > Upload File
df_spark = spark.read.parquet("/Workspace/Users/jshanodaniel@gmail.com/merged_feature_engineered steam data.parquet")

# Convert to Pandas (required because PPS sampling is row-level)
df = df_spark.toPandas()

# ---------------------------------------------------------
# 2. BASIC CLEANING
# ---------------------------------------------------------
num_cols = [
    "author_playtime_forever",
    "recommendations_total",
    "mat_final_price"
]

df[num_cols] = df[num_cols].fillna(0)

# ---------------------------------------------------------
# 3. CREATE REVIEW YEAR IF NOT PRESENT
# ---------------------------------------------------------
if "review_year" not in df.columns:
    df["timestamp_created"] = pd.to_datetime(df["timestamp_created"], errors="coerce")
    df["review_year"] = df["timestamp_created"].dt.year

# ---------------------------------------------------------
# 4. DEFINE STRATA
# ---------------------------------------------------------
strata_columns = ["type", "is_free", "language", "review_year", "voted_up"]

df[strata_columns] = df[strata_columns].astype("category")

# ---------------------------------------------------------
# 5. PPS WEIGHTS
# ---------------------------------------------------------
def compute_pps_weight(group):
    weight = (
        group["author_playtime_forever"] +
        group["recommendations_total"] +
        group["mat_final_price"]
    )
    weight = weight.replace(0, 1)
    return weight / weight.sum()

df["pps_weight"] = df.groupby(strata_columns, group_keys=False).apply(compute_pps_weight)

# ---------------------------------------------------------
# 6. SAMPLING PARAMETERS
# ---------------------------------------------------------
SAMPLE_FRACTION = 0.10
MIN_SAMPLES_PER_STRATUM = 10
RANDOM_STATE = 42

# ---------------------------------------------------------
# 7. MULTISTAGE SAMPLING
# ---------------------------------------------------------
sampled_df = []

for stratum_values, group in df.groupby(strata_columns):
    n = int(len(group) * SAMPLE_FRACTION)
    n = max(n, MIN_SAMPLES_PER_STRATUM)

    if n >= len(group):
        sampled_df.append(group)
        continue

    sample = group.sample(
        n=n,
        weights="pps_weight",
        replace=False,
        random_state=RANDOM_STATE
    )
    
    sampled_df.append(sample)

final_sample = pd.concat(sampled_df).reset_index(drop=True)

# ---------------------------------------------------------
# 8. SAVE SAMPLE BACK TO DBFS
# ---------------------------------------------------------
sample_spark = spark.createDataFrame(final_sample)
sample_spark.write.mode("overwrite").parquet("/Workspace/Users/jshanodaniel@gmail.com/merged_feature_sample_engineered_steam_data.parquet")

print("Sampling complete.")
print("Sample size:", final_sample.shape)

Sampling complete.
Sample size: (101596, 71)
